# Method 1 (Traditional): TF-IDF + Logistic Regression
**Owner:** Jana Algahtani — Group 7, Fake News Detection (ISOT)

1. loads Arwa's `train.csv`, `dev.csv`, `test.csv`,
2. tunes `ngram_range` and `C` on the **dev** set only (macro F1),
3. saves the best model trained on `input`,
4. trains the final model on `input_clean` for the **Try it** cell,
5. Try it: predicts with the saved model, no retraining, handles empty input.

The test set is **not** used here; Method 1's test results are computed in the Results section.

## Part 1 — Setup

In [1]:
import random, re, sys, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn, joblib
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

SPLIT_DIR = Path("data/splits")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

print("Python      ", sys.version.split()[0])
print("pandas      ", pd.__version__)
print("scikit-learn", sklearn.__version__)

Python       3.13.18
pandas       2.2.3
scikit-learn 1.6.1


### `remove_cues` (copy for running this part alone)

In [2]:
# Only needed when running Method 1 alone. In the merged notebook,
# remove_cues() comes from the Data section: delete this cell there.
if "remove_cues" not in globals():
    _DATELINE   = re.compile(r"^[^\n]{0,150}?\(\s*reuters\s*\)\s*[-\u2013\u2014]*\s*", re.I)
    _URL        = re.compile(r"(https?://\S+|www\.\S+|pic\.twitter\.com/\S+|\bbit\.ly/\S+|\bt\.co/\S+)", re.I)
    _HANDLE     = re.compile(r"@\w+")
    _IMG_CREDIT = re.compile(r"(featured image|header image|images?|photos?|screenshot|screen capture|screengrab)"
                             r"\s*(via|by|credit|credits|courtesy of|from)\b[^.\n]*", re.I)
    _SITE_NAMES = re.compile(r"getty\s*images|\b(?:21st century wire|21wire|reuters)\b", re.I)
    _MEDIA_TAG  = re.compile(r"[\(\[]\s*(video|videos|tweet|tweets|watch|image|images|photo|photos|audio|"
                             r"screenshot|screenshots|details|graphic video)\s*[\)\]]", re.I)
    # Reuters note on "Trump on Twitter" articles; matched in 3 parts with \s+ so extra or
    # non-breaking spaces between words do not stop the match
    _TWEET_NOTE = [
        re.compile(r"the\s+following\s+statements?\s+(?:was|were)\s+posted\s+to\s+the\s+verified\s+"
                   r"twitter\s+accounts?\s+of\s+(?:u\.\s*s\.\s+)?(?:president(?:-elect)?\s+)?donald\s+(?:j\.\s+)?trump"
                   r"(?:\s*,?\s*(?:and\s+)?@\w+)*\s*\.?", re.I),
        re.compile(r"the\s+opinions?\s+expressed\s+(?:are|is)\s+(?:his|her|their)\s+own\s*\.?", re.I),
        re.compile(r"(?:reuters\s+)?has\s+not\s+edited\s+the\s+statements?\s+or\s+confirmed\s+(?:their|its)\s+accuracy\s*\.?", re.I),
    ]
    _CREDITS    = re.compile(r"\(\s*(?:additional\s+)?(?:reporting|writing|editing|compiled|this story|this version)\b[^)]*\)", re.I)
    _EMPTY_PAR  = re.compile(r"\(\s*\)|\[\s*\]")
    _SPACES     = re.compile(r"\s+")

    def remove_cues(text: str) -> str:
        """Remove source-identifying cues from a title or article body.
        Repeated until nothing changes, because removing one cue can bring two
        parts of another cue together, e.g. "Image (VIDEO) via YouTube"."""
        if not isinstance(text, str):
            return ""
        t = _DATELINE.sub("", text)
        for _ in range(5):
            before = t
            for rx in _TWEET_NOTE:
                t = rx.sub(" ", t)
            t = _CREDITS.sub(" ", t)
            t = _URL.sub(" ", t)
            t = _HANDLE.sub(" ", t)
            t = _IMG_CREDIT.sub(" ", t)
            t = _MEDIA_TAG.sub(" ", t)
            t = _SITE_NAMES.sub(" ", t)
            t = _EMPTY_PAR.sub(" ", t)
            t = _SPACES.sub(" ", t)
            if t == before:
                break
        return _SPACES.sub(" ", t).strip()

    def first_words(text: str, n: int = 200) -> str:
        return " ".join(str(text).split()[:n])

    def make_input(title: str, body: str) -> str:
        """Model input = title + first N_WORDS words of the body (same for both methods)."""
        return (str(title).strip() + ". " + first_words(body)).strip()
    print("remove_cues loaded (copy of the Data section)")

remove_cues loaded (copy of the Data section)


## Part 2 — Load Arwa's splits
In the merged notebook the files already exist (the Data section makes them).
Running this part alone: upload `splits.zip` to the Colab session first; it is unzipped here.

In [3]:
if not (SPLIT_DIR / "train.csv").exists() and Path("splits.zip").exists():
    with zipfile.ZipFile("splits.zip") as zf:
        zf.extractall(".")
    print("Unzipped splits.zip")

train = pd.read_csv(SPLIT_DIR / "train.csv")
dev   = pd.read_csv(SPLIT_DIR / "dev.csv")
test  = pd.read_csv(SPLIT_DIR / "test.csv")

for name, part in [("train", train), ("dev", dev), ("test", test)]:
    print(f"{name:5s} {len(part):6d} rows | Fake share {part.label.mean():.1%}")

X_train, y_train = train["input"], train["label"]
X_dev,   y_dev   = dev["input"],   dev["label"]

Unzipped splits.zip
train  27004 rows | Fake share 45.1%
dev     5787 rows | Fake share 45.1%
test    5787 rows | Fake share 45.1%


## Part 3 — Tune `ngram_range` and `C` on the dev set
TF-IDF (lowercase, terms in at least 2 documents, sublinear tf) + Logistic Regression.
Grid: `ngram_range` ∈ {(1,1), (1,2)} × `C` ∈ {0.1, 1, 10, 100, 1000, 10⁴, 10⁵}. Selected by **dev macro F1**; on a tie the smaller C is kept.
C = 10⁴ and 10⁵ were added to check that the best C is not at the edge of the grid.

In [4]:
def make_model(ngram, C):
    return Pipeline([
        ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=ngram,
                                  min_df=2, sublinear_tf=True)),
        ("lr", LogisticRegression(C=C, max_iter=2000, random_state=SEED)),
    ])

results = []
for ngram in [(1, 1), (1, 2)]:
    for C in [0.1, 1, 10, 100, 1000, 10000, 100000]:
        pred = make_model(ngram, C).fit(X_train, y_train).predict(X_dev)
        results.append({"ngram": str(ngram), "C": C,
                        "dev_acc": accuracy_score(y_dev, pred),
                        "dev_macro_f1": f1_score(y_dev, pred, average="macro")})
        print(f"ngram={ngram}  C={C:<6}  dev macro F1={results[-1]['dev_macro_f1']:.4f}")

# ties -> the smaller C (simpler model) wins
results = (pd.DataFrame(results)
           .sort_values(["dev_macro_f1", "C"], ascending=[False, True], kind="mergesort")
           .reset_index(drop=True))
results.round(4)

ngram=(1, 1)  C=0.1     dev macro F1=0.9645
ngram=(1, 1)  C=1       dev macro F1=0.9873
ngram=(1, 1)  C=10      dev macro F1=0.9920
ngram=(1, 1)  C=100     dev macro F1=0.9937
ngram=(1, 1)  C=1000    dev macro F1=0.9941
ngram=(1, 1)  C=10000   dev macro F1=0.9941
ngram=(1, 1)  C=100000  dev macro F1=0.9941
ngram=(1, 2)  C=0.1     dev macro F1=0.9650
ngram=(1, 2)  C=1       dev macro F1=0.9850
ngram=(1, 2)  C=10      dev macro F1=0.9918
ngram=(1, 2)  C=100     dev macro F1=0.9934
ngram=(1, 2)  C=1000    dev macro F1=0.9916
ngram=(1, 2)  C=10000   dev macro F1=0.9916
ngram=(1, 2)  C=100000  dev macro F1=0.9916


,ngram,C,dev_acc,dev_macro_f1
0,"(1, 1)",1000.0,0.9941,0.9941
1,"(1, 1)",10000.0,0.9941,0.9941
2,"(1, 1)",100000.0,0.9941,0.9941
3,"(1, 1)",100.0,0.9938,0.9937
4,"(1, 2)",100.0,0.9934,0.9934
5,"(1, 1)",10.0,0.9921,0.9920
6,"(1, 2)",10.0,0.9919,0.9918
7,"(1, 2)",1000.0,0.9917,0.9916
8,"(1, 2)",10000.0,0.9917,0.9916
9,"(1, 2)",100000.0,0.9917,0.9916


## Part 4 — Save the best model (trained on `input`)

In [5]:
best = results.iloc[0]
BEST_NGRAM, BEST_C = eval(best["ngram"]), float(best["C"])
print(f"Best: ngram_range={BEST_NGRAM}, C={BEST_C}, dev macro F1={best['dev_macro_f1']:.4f}")

best_model = make_model(BEST_NGRAM, BEST_C).fit(X_train, y_train)
joblib.dump(best_model, MODEL_DIR / "method1_tfidf_lr.joblib")

Best: ngram_range=(1, 1), C=1000.0, dev macro F1=0.9941


['models/method1_tfidf_lr.joblib']

## Part 5 — Final model on cue-removed text (used by Try it)
`ngram_range` and `C` are tuned again, this time on `input_clean` (train → dev), the same way as Method 2's final model.

In [6]:
X_train_c, X_dev_c = train["input_clean"], dev["input_clean"]

results_clean = []
for ngram in [(1, 1), (1, 2)]:
    for C in [0.1, 1, 10, 100, 1000, 10000, 100000]:
        pred = make_model(ngram, C).fit(X_train_c, y_train).predict(X_dev_c)
        results_clean.append({"ngram": str(ngram), "C": C,
                              "dev_acc": accuracy_score(y_dev, pred),
                              "dev_macro_f1": f1_score(y_dev, pred, average="macro")})
        print(f"ngram={ngram}  C={C:<6}  dev macro F1 (clean)={results_clean[-1]['dev_macro_f1']:.4f}")

# ties -> the smaller C (simpler model) wins
results_clean = (pd.DataFrame(results_clean)
                 .sort_values(["dev_macro_f1", "C"], ascending=[False, True], kind="mergesort")
                 .reset_index(drop=True))
best_c = results_clean.iloc[0]
CLEAN_NGRAM, CLEAN_C = eval(best_c["ngram"]), float(best_c["C"])
print(f"\nBest on clean text: ngram_range={CLEAN_NGRAM}, C={CLEAN_C}, dev macro F1={best_c['dev_macro_f1']:.4f}")

final_model = make_model(CLEAN_NGRAM, CLEAN_C).fit(X_train_c, y_train)
joblib.dump(final_model, MODEL_DIR / "method1_final_clean.joblib")
results_clean.round(4)

ngram=(1, 1)  C=0.1     dev macro F1 (clean)=0.9540
ngram=(1, 1)  C=1       dev macro F1 (clean)=0.9726
ngram=(1, 1)  C=10      dev macro F1 (clean)=0.9803
ngram=(1, 1)  C=100     dev macro F1 (clean)=0.9824
ngram=(1, 1)  C=1000    dev macro F1 (clean)=0.9827
ngram=(1, 1)  C=10000   dev macro F1 (clean)=0.9825
ngram=(1, 1)  C=100000  dev macro F1 (clean)=0.9824
ngram=(1, 2)  C=0.1     dev macro F1 (clean)=0.9545
ngram=(1, 2)  C=1       dev macro F1 (clean)=0.9752
ngram=(1, 2)  C=10      dev macro F1 (clean)=0.9827
ngram=(1, 2)  C=100     dev macro F1 (clean)=0.9853
ngram=(1, 2)  C=1000    dev macro F1 (clean)=0.9864
ngram=(1, 2)  C=10000   dev macro F1 (clean)=0.9860
ngram=(1, 2)  C=100000  dev macro F1 (clean)=0.9860

Best on clean text: ngram_range=(1, 2), C=1000.0, dev macro F1=0.9864


,ngram,C,dev_acc,dev_macro_f1
0,"(1, 2)",1000.0,0.9865,0.9864
1,"(1, 2)",10000.0,0.9862,0.9860
2,"(1, 2)",100000.0,0.9862,0.9860
3,"(1, 2)",100.0,0.9855,0.9853
4,"(1, 2)",10.0,0.9829,0.9827
5,"(1, 1)",1000.0,0.9829,0.9827
6,"(1, 1)",10000.0,0.9827,0.9825
7,"(1, 1)",100.0,0.9825,0.9824
8,"(1, 1)",100000.0,0.9825,0.9824
9,"(1, 1)",10.0,0.9805,0.9803


## Part 6 — Try it
Loads the saved model (no retraining). The user's text goes through `remove_cues()` first (from the Data section), because the final model was trained on clean text.

In [7]:
tryit_model = joblib.load(MODEL_DIR / "method1_final_clean.joblib")

def try_it(text):
    if text is None or str(text).strip() == "":
        print("Please enter some news text (a headline and/or article).")
        return
    clean = remove_cues(str(text))
    if clean == "":
        print("Nothing left to classify after removing links and source tags.")
        return
    p_fake = tryit_model.predict_proba([clean])[0][1]
    label = "Fake" if p_fake >= 0.5 else "Real"
    prob = p_fake if label == "Fake" else 1 - p_fake
    print(f"{label}  (probability {prob:.2f})")

try_it("")
try_it("Pope Francis endorses Donald Trump for president, shocking the world")
try_it("The Federal Reserve kept interest rates unchanged on Wednesday, citing steady job growth")

Please enter some news text (a headline and/or article).
Fake  (probability 0.97)
Real  (probability 1.00)


Type your own text:

In [8]:
my_text = "The central bank raised interest rates by a quarter point on Thursday, citing rising inflation"
try_it(my_text)

Real  (probability 1.00)


## Original labelled examples (Jana)
Two short texts written by Jana for this project (not taken from ISOT or any news site), shown separately from the test results.
They are run through the Try it model.

In [9]:
jana_examples = [
    ("Fake", "SHOCKING: Scientists Admit Drinking Coffee Before Bed Makes You Immune To The Flu, "
             "And Big Pharma Is Furious. A leaked report that the mainstream media refuses to show you "
             "proves that one cup of coffee at night completely blocks the flu virus. Doctors have known "
             "this for years but kept it secret to protect vaccine profits. Share this before it gets deleted!"),
    ("Real", "City council approves budget for new public library. The city council voted 7-2 on Tuesday "
             "to approve a $12 million budget for a new public library in the downtown area, officials said. "
             "Construction is expected to begin next spring and take about 18 months. The council said the "
             "project will be funded through a mix of state grants and existing city funds."),
]
for true_label, text in jana_examples:
    print(f"True label: {true_label}  ->  predicted: ", end="")
    try_it(text)

True label: Fake  ->  predicted: Fake  (probability 1.00)
True label: Real  ->  predicted: Real  (probability 1.00)
